# <div align="center">🌟 مدار (Madar) - Google Colab Quickstart</div>
### <div align="center">AI-Native, High-Performance Dialect & Runtime for Ring</div>

[![Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com) 
[![GitHub](https://img.shields.io/badge/GitHub-Madar-blue.svg)](https://github.com/azzeddine2017/Madar) 
[![AVX2](https://img.shields.io/badge/CPU-AVX2%20%7C%20FMA-green.svg)]() 
[![OpenCL](https://img.shields.io/badge/GPU-OpenCL%20Zero--Copy-purple.svg)]()

---

مرحباً بك في دفتر التشغيل السحابي لبيئة **مدار (Madar)**.

تم تصميم هذا الدفتر لاستغلال الإمكانيات العالية لسحابة **Google Colab**:
1. **المعالج (CPU):** تسريع العمليات المتجهية عبر تعليمات `AVX2` و `FMA`.
2. **كارت الشاشة (GPU):** تسريع طبقات التغذية الأمامية (FFN) واستنتاج النماذج عبر **NVIDIA OpenCL Zero-Copy**.
3. **نماذج ذكية وكبيرة:** تحميل نماذج استدلال منطقي متقدمة مثل **DeepSeek-R1-7B** و **Qwen2.5-7B**.
4. **مكتبة القلم (AlQalam C++20):** تسريع البحث الهجين (Hybrid RAG) وحفظ واسترجاع المتجهات بالقرص.
5. **التفاعل المباشر عبر `%%madar`:** كتابة وتنفيذ كود مدار داخل أي خلية كود في الدفتر بسلاسة تامة!

## ⚙️ الخطوة 1: فحص بيئة العتاد (Hardware Specs)
نتحقق من دعم المعالج لـ AVX2 وتوفر كارت الشاشة (NVIDIA T4 / V100 / A100).

In [ ]:
# فحص المعالج وتوفر تقنيات AVX2 و FMA
!lscpu | grep -E "Model name|avx2|fma"

# فحص بطاقة الرسوميات GPU
!nvidia-smi

## 🔨 الخطوة 2: تثبيت الاعتماديات وبناء مدار ومكتبة القلم من المصدر
نقوم باستنساخ مستودع مدار وبناء النواة بالكامل ومكتبة **AlQalam C++20** والامتدادات عبر GCC بأعلى مستويات التحسين (`-O3 -mavx2 -mfma -mf16c -pthread`).

In [ ]:
import os, sys

# 1. استنساخ المشروع إذا لم يكن موجوداً أو التحديث إذا كان موجوداً
if not os.path.exists("/content/Madar/language"):
    !git clone https://github.com/azzeddine2017/Madar.git /content/Madar
    %cd /content/Madar
else:
    %cd /content/Madar
    !git pull

# 2. تثبيت الحزم المطلوبة لـ OpenCL و GCC و G++ و Poppler
!apt-get update -qq && apt-get install -y -qq build-essential g++ ocl-icd-opencl-dev curl libcurl4-openssl-dev poppler-utils

# 3. بناء نواة مدار ومكتبة القلم AlQalam والامتدادات دفعة واحدة
!bash tools/build_linux.sh

# 4. التحقق من الإصدار والمواصفات
!bin/ring -v
!bin/ring --sysinfo

### 🧪 تشغيل حزمة الاختبارات الشاملة (Automated Test Suite)
تشغيل اختبارات الوحدة للنواة ومحرك التنسور والذكاء الاصطناعي للتحقق من سلامة البيئة بنسبة 100%:

In [ ]:
!bash tests/run_all.sh

## 🪄 الخطوة 3: تفعيل الأمر السحري `%%madar` و `%%ring`
يتيح لك هذا السكربت كتابة كود مدار أو رينج مباشرة في أي خلية كود تبدأ بـ `%%madar` وتشغيلها فوراً!

In [ ]:
from IPython.core.magic import register_cell_magic
import subprocess, tempfile, os, sys

MADAR_BIN = os.path.abspath("bin/ring")

@register_cell_magic
def madar(line, cell):
    """IPython Cell Magic لتنفيذ كود مدار وعرض المخرجات مباشرة"""
    with tempfile.NamedTemporaryFile("w", suffix=".ring", delete=False, encoding="utf-8") as f:
        f.write(cell)
        tmp_file = f.name
    
    try:
        env = os.environ.copy()
        env["LD_LIBRARY_PATH"] = os.path.abspath("lib") + ":" + env.get("LD_LIBRARY_PATH", "")
        cwd = os.path.abspath("/content/Madar") if os.path.exists("/content/Madar") else os.path.abspath(".")
        res = subprocess.run([MADAR_BIN, tmp_file], capture_output=True, text=True, env=env, cwd=cwd)
        if res.stdout:
            print(res.stdout, end="")
        if res.stderr:
            print(res.stderr, end="", file=sys.stderr)
    finally:
        if os.path.exists(tmp_file):
            os.remove(tmp_file)

# تسجيل الاسم البديل %%ring أيضاً
ip = get_ipython()
ip.register_magic_function(madar, 'cell', 'ring')
print("✅ تم تسجيل الأمر السحري %%madar و %%ring بنجاح! يمكنك الآن كتابة كود مدار في أي خلية.")

### 🧪 تجربة سريعة للتركيبات اللغوية الحديثة والمحرك المتجهي عبر `%%madar`

In [ ]:
%%madar
# 1. السلاسل النصية المهيأة (f-strings)
cProject = "مدار (Madar)"
nVersion = 0.3
see f"مرحباً بكم في {cProject} الإصدار {nVersion} على Google Colab!" + nl

# 2. اشتمال القوائم (List Comprehensions)
aSquares = [x*x for x in 1:10 if x % 2 = 0]
see "مربعات الأعداد الزوجية: ["
for i = 1 to len(aSquares)
    see "" + aSquares[i]
    if i < len(aSquares) see ", " ok
next
see "]" + nl

# 3. التقسيم المتقدم للمصفوفات والسلاسل (Advanced Slicing)
cText = "GoogleColabWithMadar"
see f"عكس النص: {cText[::-1]}" + nl

# 4. الحلقات السريعة فائقة الأداء عبر مترجم JIT (Native Machine Code)
t0 = clock()
nSum = 0
dour i = 1 to 1000000
    nSum += i
next
t1 = clock()
see f"حساب المليون عبر JIT في: {(t1 - t0) / clockspersecond()} ثانية! المجموع: {nSum}" + nl

## 🧠 الخطوة 4: تحميل نماذج الذكاء الاصطناعي القوية (7B Reasoning Models)
بما أن بيئة Colab تمنحنا قرابة **12.7 GB RAM** و **15 GB GPU VRAM**، فإننا نستهدف النماذج عالية الذكاء:
- **DeepSeek-R1-Distill-Qwen-7B:** نموذج التفكير العميق المنطقي (Chain-of-Thought) وحل المسائل البرمجية والمعقدة (~4.68 GB).
- **Qwen2.5-7B-Instruct:** النموذج القياسي المتميز في اللغة العربية والتعليمات والترميز (~4.35 GB).
- **BGE-M3:** النموذج الرائد عالمياً في البحث الدلالي متعدد اللغات ونظام RAG (~1.19 GB).

اختر النموذج الذي تريده وشغّل الخلية التالية (سرعة التحميل في كولاب تصل إلى 80-100 ميجابايت/ثانية):

In [ ]:
# تحميل نموذج التفكير المنطقي الفائق DeepSeek-R1-7B
!bash tools/download_colab_models.sh deepseek7b

# (اختياري) إذا أردت نموذج البحث الدلالي BGE-M3 لتقنية RAG، قم بإلغاء التعليق:
# !bash tools/download_colab_models.sh bge

## ⚡ الخطوة 5: تشغيل استدلال الذكاء الاصطناعي مع تسريع كارت الشاشة (GPU OpenCL)
نقوم بتحميل النموذج وتفعيل كارت الشاشة `model.setGPU(1)` للاستفادة القصوى من الذاكرة الصفرية (Zero-Copy) في طبقات FFN، مع إرسال سؤال يختبر قدرات التفكير المعقدة.

In [ ]:
%%madar
load "stdlibcore.ring"
load "libraries/madar_ai/madar_ai.ring"

# تحديد مسار النموذج
cModelPath = "models/deepseek-r1-distill-qwen-7b-q4_0.gguf"
if not fexists(cModelPath)
    cModelPath = "models/qwen2.5-7b-instruct-q4_0.gguf"
ok

see f"[MADAR] جارٍ تحميل النموذج إلى الذاكرة: {cModelPath}" + nl
t0 = clock()
oModel = ai_open(cModelPath)
t1 = clock()
see f"[MADAR] تم تحميل النموذج في: {(t1 - t0) / clockspersecond()} ثانية عبر Zero-Copy mmap!" + nl

# تفعيل خيوط المعالجة المتوازية (Colab CPUs) وكارت الشاشة (GPU)
oModel.setThreads(4)
oModel.setGPU(1)
oModel.setTemperature(0.6)
oModel.setMaxTokens(256)

cPrompt = "حل هذه المسألة بخطوات تفكير منطقية: قطار يسير بسرعة 120 كم/ساعة، قطع مسافة 300 كم، كم دقيقة استغرقها؟"
see nl + "========================================================" + nl
see "السؤال: " + cPrompt + nl
see "========================================================" + nl

tGen0 = clock()
cAnswer = oModel.generate(cPrompt, 256)
tGen1 = clock()

see nl + cAnswer + nl
see "========================================================" + nl
see f"سرعة التوليد: {oModel.tokensPerSecond()} رمز/ثانية (زمن التوليد: {(tGen1 - tGen0)/clockspersecond()} ثانية)" + nl

## ✒️ الخطوة 6: البحث الهجين واسترجاع المعرفة عبر مكتبة القلم (AlQalam Hybrid RAG)
تدمج مكتبة **AlQalam** المبنية بلغة C++20 بين البحث الدلالي بالمتجهات (70%) والبحث اللفظي بالفهرس المعكوس `QalamIndex` (30%) مع الحفظ الثنائي فائق السرعة لقاعدة المعرفة على القرص.

In [ ]:
%%madar
load "stdlibcore.ring"
load "libraries/madar_ai/madar_ai.ring"
load "libraries/madar_ai/rag.ring"
load "alqalam.ring"

cModelPath = "models/deepseek-r1-distill-qwen-7b-q4_0.gguf"
if not fexists(cModelPath) cModelPath = "models/qwen2.5-7b-instruct-q4_0.gguf" ok

oTimer = new QalamChronos()
oTimer.reset()

see "[RAG] تحميل النموذج اللغوي لتقنية الـ RAG... " + nl
llm = ai_open(cModelPath)
llm.setGPU(1)
llm.setThreads(4)
store = new VectorStore(llm)

# إضافة وثائق تجريبية وفهرستها عبر AlQalam
store.addText("مشروع مدار يدعم تسريع المعالجات الرسومية عبر OpenCL وحسابات AVX2 الخارقة.")
store.addText("مكتبة القلم AlQalam توفر هياكل بيانات C++ سريعة جداً مثل الفهرس المعكوس QalamIndex والمتجهات المتصلة.")
store.addText("عاصمة الجمهورية العربية السورية هي دمشق وتعد من أقدم العواصم المأهولة في العالم.")

cQuestion = "ما هي مميزات مكتبة القلم AlQalam في مدار؟"
see nl + "السؤال: " + cQuestion + nl

# تنفيذ البحث الهجين (70% دلالي + 30% لفظي بـ AlQalam)
oTimer.reset()
aResults = store.hybridSearch(cQuestion, 2, 0.7)
see f"[RAG] نتائج البحث الهجين (زمن البحث: {oTimer.elapsed()}):" + nl

for res in aResults
    see f"  - [تطابق: {res[1]}] (دلالي: {res[3]} | لفظي: {res[4]}) {res[2].cText}" + nl
next

best_text = aResults[1][2].cText
see nl + "[RAG] توليد الإجابة الحتمية بالاستناد للوثيقة المسترجعة:" + nl
cPrompt = f"أجب باختصار شديد باللغة العربية بناءً على هذا المرجع: {best_text}\nالسؤال: {cQuestion}"
see llm.generate(cPrompt, 256) + nl

## 🤖 الخطوة 7: تشغيل الوكيل الذاتي (Autonomous Agent) واستدعاء الأدوات
وكيل مدار الذاتي يستطيع استدعاء أدوات لغة مدار تلقائياً (الحساب الدقيق، قراءة الملفات، الذاكرة المتجهية، والتشخيص) ثم تركيب الإجابة بدقة وحفظ الحقائق في الذاكرة الدائمة.

In [ ]:
%%madar
load "stdlibcore.ring"
load "libraries/madar_ai/agent.ring"

cModelPath = "models/deepseek-r1-distill-qwen-7b-q4_0.gguf"
if not fexists(cModelPath)
    cModelPath = "models/qwen2.5-7b-instruct-q4_0.gguf"
ok

oAgentModel = ai_open(cModelPath)
oAgentModel.setGPU(1)
oAgentModel.setThreads(4)
oAgent = new AIAgent(oAgentModel)
oAgent.remember("مشروع مدار هو بيئة تشغيل فائقة السرعة للغة رينج مصممة خصيصاً للذكاء الاصطناعي")
oAgent.remember("تم بناء مدار بلغة C القياسية مع تسريع AVX2 و OpenCL لدعم استنتاج النماذج محلياً")

see "[AGENT] الذاكرة المتجهية جاهزة! نقوم الآن بسؤال الوكيل:" + nl

cQuery = "احسب ناتج ضرب 4589 في 321 باستخدام أداة الحساب calc، واذكر لي نبذة عن مشروع مدار من ذاكرتك"
see nl + "المستخدم > " + cQuery + nl + nl

cResponse = oAgent.chat(cQuery)
see nl + "الوكيل الذكي > " + nl + cResponse + nl

## 💬 الخطوة 8: واجهة محادثة تفاعلية حية (Interactive Chat Widget)
يمكنك التحدث مع النموذج والوكيل الذكي مباشرة من داخل كولاب عبر صندوق إدخال نصي تفاعلي وأزرار تحكم مريحة.

In [ ]:
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
import subprocess, os, tempfile

chat_history = []

out_area = widgets.Output()
text_input = widgets.Text(
    placeholder='اكتب سؤالك لنموذج مدار هنا واضغط Enter...',
    layout=widgets.Layout(width='75%')
)
send_btn = widgets.Button(
    description='إرسال 🚀',
    button_style='primary',
    layout=widgets.Layout(width='20%')
)

def ask_madar(prompt):
    ring_code = f'''
load "libraries/madar_ai/madar_ai.ring"
cModel = "models/deepseek-r1-distill-qwen-7b-q4_0.gguf"
if not fexists(cModel) cModel = "models/qwen2.5-7b-instruct-q4_0.gguf" ok
model = ai_open(cModel)
model.setGPU(1)
model.setThreads(4)
model.setTemperature(0.7)
model.setMaxTokens(200)
see model.generate("{prompt}", 200)
'''
    with tempfile.NamedTemporaryFile("w", suffix=".ring", delete=False, encoding="utf-8") as f:
        f.write(ring_code)
        tmp_name = f.name
    
    try:
        env = os.environ.copy()
        env["LD_LIBRARY_PATH"] = os.path.abspath("lib") + ":" + env.get("LD_LIBRARY_PATH", "")
        cwd = os.path.abspath("/content/Madar") if os.path.exists("/content/Madar") else os.path.abspath(".")
        res = subprocess.run([os.path.abspath("bin/ring"), tmp_name], capture_output=True, text=True, env=env, cwd=cwd)
        return res.stdout.strip()
    finally:
        if os.path.exists(tmp_name):
            os.remove(tmp_name)
            
def on_send(b):
    user_msg = text_input.value.strip()
    if not user_msg:
        return
    text_input.value = ''
    
    with out_area:
        display(HTML(f'<div style="background-color:#e3f2fd; padding:10px; border-radius:8px; margin:5px 0;"><b>🧑 أنت:</b> {user_msg}</div>'))
        display(HTML('<div id="loading" style="color:#888;"><i>⏳ مدار يفكر...</i></div>'))
    
    bot_msg = ask_madar(user_msg)
    
    with out_area:
        clear_output(wait=True)
        for role, text in chat_history:
            bg = '#e3f2fd' if role == 'user' else '#f1f8e9'
            author = '🧑 أنت' if role == 'user' else '🪐 مدار'
            display(HTML(f'<div style="background-color:{bg}; padding:10px; border-radius:8px; margin:5px 0;"><b>{author}:</b> {text}</div>'))
        
        display(HTML(f'<div style="background-color:#f1f8e9; padding:10px; border-radius:8px; margin:5px 0;"><b>🪐 مدار:</b> {bot_msg}</div>'))
    
    chat_history.append(('user', user_msg))
    chat_history.append(('bot', bot_msg))

send_btn.on_click(on_send)
text_input.on_submit(on_send)

controls = widgets.HBox([text_input, send_btn])
display(widgets.VBox([out_area, controls]))